# 01-preprocess — Tiền xử lý ảnh

**Câu hỏi**: DINO thực sự nhận đầu vào gì, lúc fine-tune và lúc trích đặc trưng; tiền xử lý làm vết lỗi mờ đi bao nhiêu? (Q03.03)

### Nguồn gốc (provenance)

In commit và trạng thái dirty của code, rồi đặt `XLA_FLAGS` trước mọi lần import jax (mẫu ở `lab/README.vi.md` §11).

In [1]:
# cell 0: provenance (run first)
import os, subprocess, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path[:0] = [str(ROOT), str(ROOT / "lab")]
import replay; os.environ["XLA_FLAGS"] = replay._xla_flags()   # GPU deterministic mode before any jax import
git = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.strip()
print("commit", git("rev-parse", "--short", "HEAD"),
      "| code dirty", bool(git("status", "--porcelain", "--", ".", ":(exclude)runs", ":(exclude)notes",
                               ":(exclude)lab/*.ipynb", ":(exclude)lab/*.md")),
      "| XLA_FLAGS", os.environ["XLA_FLAGS"])

commit d67fdec | code dirty False | XLA_FLAGS --xla_gpu_deterministic_ops=true


- **Kết quả**: commit `d67fdec`, code dirty False, XLA_FLAGS `--xla_gpu_deterministic_ops=true`.

## 1. Code

### 1.1 Tham số phase 1 và đoạn code tiền xử lý

Đọc `config.json` của 8 run phase 1 (GC10 và NEU; mỗi bộ gồm `fz` và `ft` r0–r2): commit, dirty, cách resize, có fine-tune không, `FT_SEED`, mã thoát. Kiểm thư mục `feature_extractor/` không đổi từ commit các run đã chạy, rồi in đúng đoạn tạo transform trong `feature_extractor/dino-neudet.py`. Không run nào truyền `--img_size`, nên `--resize full` co ảnh về 224×224 (giá trị mặc định).

In [2]:
# cell 1: the 8 phase-1 runs (config.json) and the preprocessing code they ran
import json, re
import pandas as pd
SRC = Path("feature_extractor/dino-neudet.py")
runs = sorted(d for d in Path("runs").glob("*__*_f[tz]_cls*first_img80pct*") if d.is_dir() and "_km_" not in d.name)
rows = []
for d in runs:
    c = json.loads((d / "config.json").read_text())
    cmd = " ".join(c["command"])
    get = lambda pat, default: (re.search(pat, cmd) or [None, default])[1]
    rows.append(dict(tag=d.name.split("__")[1], commit=c["git"]["commit"][:7], dirty=c["git"]["dirty"],
                     resize=get(r"--resize (\S+)", "crop (default)"), finetune="--no_finetune" not in cmd,
                     FT_SEED=get(r"FT_SEED=(\d+)", "-"), exit=c.get("exit_code")))
print(pd.DataFrame(rows).to_string(index=False))
assert git("cat-file", "-t", "97c57ee") == "commit", "97c57ee is not in the history (shallow clone?)"
changed = git("diff", "--name-only", "97c57ee", "HEAD", "--", "feature_extractor")
print("feature_extractor/ changed since 97c57ee:", changed or "nothing")
text = SRC.read_text().splitlines()
i = next(k for k, l in enumerate(text) if l.lstrip().startswith("resize = ("))
print(f"\n{SRC}, lines {i + 1}-{i + 5}:", *text[i:i + 5], sep="\n")
assert len(rows) == 8 and {r["commit"] for r in rows} == {"97c57ee"} and not any(r["dirty"] for r in rows) and not changed

                          tag  commit  dirty         resize  finetune FT_SEED  exit
   gc10_fz_cls5first_img80pct 97c57ee  False           full     False       -     0
gc10_ft_cls5first_img80pct_r0 97c57ee  False           full      True       0     0
gc10_ft_cls5first_img80pct_r1 97c57ee  False           full      True       1     0
gc10_ft_cls5first_img80pct_r2 97c57ee  False           full      True       2     0
    neu_fz_cls3first_img80pct 97c57ee  False crop (default)     False       -     0
 neu_ft_cls3first_img80pct_r0 97c57ee  False crop (default)      True       0     0
 neu_ft_cls3first_img80pct_r1 97c57ee  False crop (default)      True       1     0
 neu_ft_cls3first_img80pct_r2 97c57ee  False crop (default)      True       2     0
feature_extractor/ changed since 97c57ee: nothing

feature_extractor/dino-neudet.py, lines 69-73:
    resize = ([torchvision.transforms.Resize(tuple(a.img_size), 3)] if a.resize == "full" else          # whole image
              [torchvision.t

- **Kết quả**: cả 8 run phase 1 chạy ở commit `97c57ee`, không dirty, mã thoát 0:
  - GC10 dùng `--resize full`, NEU dùng `crop` (mặc định);
  - `fz` không fine-tune; `ft` r0, r1, r2 fine-tune với `FT_SEED` 0, 1, 2.

  Thư mục `feature_extractor/` không đổi từ `97c57ee`. Transform ở dòng 69–73 của `dino-neudet.py`:
  - `full`: `Resize((224, 224), 3)`;
  - `crop`: `Resize(256, 3)` rồi `CenterCrop(224)`;
  - sau đó cả hai cùng qua `ToTensor` và `Normalize` với trung bình (0.485, 0.456, 0.406) và độ lệch chuẩn (0.229, 0.224, 0.225).
- **Ý nghĩa**: code đang có trong repo chính là code đã tạo ra cả 8 bộ đặc trưng, nên đọc code này là đọc đúng tiền xử lý thật. Mã nội suy 3 là bicubic. 256 = int(224 / 0.875), tức cắt giữ 87.5% mỗi chiều như cách đánh giá chuẩn trên ImageNet. GC10 co thẳng cả ảnh về hình vuông, không cắt, nên tỉ lệ khung bị méo (`00-data-gc10` §2.1).

### 1.2 Ảnh lúc fine-tune đi qua transform nào

Đọc code bằng cây cú pháp (AST), không import: in hàm `to_dataset`, các lời gọi nó trong `main()` (một cho tập fine-tune, một cho tập trích đặc trưng), và đếm số lần biến `transform` trong `finetune_dino` được gán và được đọc. Nếu biến này được đọc, ảnh lúc fine-tune có thể đi qua một transform khác lúc trích.

In [3]:
# cell 2: fine-tune images go through the same tfm as extraction; finetune_dino's own transform is never read
import ast
def func(path, name):
    """Syntax tree and source text of function `name` in a file, without importing it."""
    text = Path(path).read_text()
    return next(n for n in ast.walk(ast.parse(text)) if isinstance(n, ast.FunctionDef) and n.name == name), text
td, text = func(SRC, "to_dataset")
print(ast.get_source_segment(text, td))
m, _ = func(SRC, "main")
calls = [ast.get_source_segment(text, n) for n in ast.walk(m)
         if isinstance(n, ast.Call) and getattr(n.func, "id", None) == "to_dataset"]
print("\nto_dataset calls in main():", *calls, sep="\n  ")
fd, _ = func("feature_extractor/sl_finetuned_model.py", "finetune_dino")
use = {ctx: sorted(n.lineno for n in ast.walk(fd) if isinstance(n, ast.Name) and n.id == "transform" and isinstance(n.ctx, ctx))
       for ctx in (ast.Store, ast.Load)}
print(f"\nfinetune_dino (sl_finetuned_model.py): 'transform' assigned at lines {use[ast.Store]}, read at lines {use[ast.Load]}")
assert len(calls) == 2 and all(c.endswith("tfm)") for c in calls) and use[ast.Store] and not use[ast.Load]

def to_dataset(paths, labels, tfm):
    ds = Dataset.from_dict({"image": paths, "labels": labels}).cast_column("image", HFImage())
    def trans(ex):
        return {"images": [tfm(im.convert("RGB")) for im in ex["image"]], "labels": torch.tensor(ex["labels"])}
    return ds.with_transform(trans)

to_dataset calls in main():
  to_dataset([p[i] for i in keep], [l[i] for i in keep], tfm)
  to_dataset(*build(split, part), tfm)

finetune_dino (sl_finetuned_model.py): 'transform' assigned at lines [82], read at lines []


- **Kết quả**: `main()` gọi `to_dataset` hai lần, cả hai đều với `tfm`: một lần cho tập fine-tune (ảnh có nhãn của lớp cơ sở, `keep`), một lần cho tập trích đặc trưng (`build(split, part)`). `trans` chỉ trả hai khoá `images` và `labels`. Trong `finetune_dino`, biến `transform` được gán ở dòng 82 nhưng không được đọc ở dòng nào.
- **Ý nghĩa**: lúc fine-tune và lúc trích đặc trưng, DINO thấy cùng một kiểu ảnh: GC10 co thẳng về 224×224, NEU co rồi cắt giữa. Transform riêng của `finetune_dino` (Resize 256 + CenterCrop 224) là code chết. Nếu nó được dùng, ảnh GC10 lúc fine-tune sẽ bị cắt mép, khác lúc trích. Không có khoá `images2`, nên nhánh có augmentation (`FT_AUG`) không bao giờ chạy: fine-tune không dùng augmentation nào, mỗi epoch thấy đúng các ảnh như nhau, chỉ khác thứ tự (DataLoader `shuffle=True`).

## 2. Resize

### 2.1 GC10: co 2048×1000 về 224×224 làm vạch mảnh mờ đi bao nhiêu

Khi thu nhỏ, bicubic của PIL có lọc chống răng cưa (antialias): mỗi điểm ảnh ra là trung bình có trọng số của các điểm ảnh vào trong một cửa sổ rộng theo hệ số co. Hệ số co là 2048/224 ≈ 9.14 theo chiều ngang và 1000/224 ≈ 4.46 theo chiều dọc.

**Cách làm**: ảnh đen 2048×1000, vẽ một vạch trắng rộng w px (dọc hoặc ngang), co bằng đúng phép resize của `dino-neudet.py`, rồi đọc đỉnh của mặt cắt ngang qua vạch (giá trị trắng = 1). Mỗi độ rộng thử ở 10 vị trí liền nhau, vì vạch rơi vào lưới điểm ảnh ra ở các pha khác nhau. Nếu chỉ là trung bình đều, vạch hẹp hơn một điểm ảnh ra sẽ còn khoảng w / hệ số co.

In [4]:
# cell 3: bicubic downscaling 2048x1000 -> 224x224 (dino-neudet.py --resize full): what is left of a thin bright line
import numpy as np, torchvision
from PIL import Image
W, H, S = 2048, 1000, 224
resize = torchvision.transforms.Resize((S, S), 3)               # the GC10 resize of dino-neudet.py (before ToTensor)
print(resize, f"| input pixels per output pixel: x {W / S:.2f}, y {H / S:.2f}")
rows = []
for line, n, scale in (("vertical (thin in x)", W, W / S), ("horizontal (thin in y)", H, H / S)):
    for width in (1, 2, 4, 9, 18, 36):
        peak, lit = [], []
        for p in range(n // 2, n // 2 + 10):                    # 10 positions: the line meets the output grid at different phases
            a = np.zeros((H, W, 3), np.uint8)
            if line.startswith("vertical"):
                a[:, p:p + width] = 255
            else:
                a[p:p + width, :] = 255
            out = np.asarray(resize(Image.fromarray(a)))[..., 0] / 255
            prof = out[S // 2] if line.startswith("vertical") else out[:, S // 2]   # profile across the line
            peak.append(prof.max())
            lit.append(int((prof > 0).sum()))
        rows.append(dict(line=line, width_px=width, width_out=round(width / scale, 2), peak_min=round(min(peak), 3),
                         peak_max=round(max(peak), 3), out_px_lit=f"{min(lit)}-{max(lit)}"))
print(pd.DataFrame(rows).to_string(index=False))

Resize(size=(224, 224), interpolation=bicubic, max_size=None, antialias=True) | input pixels per output pixel: x 9.14, y 4.46
                  line  width_px  width_out  peak_min  peak_max out_px_lit
  vertical (thin in x)         1       0.11     0.067     0.110        1-2
  vertical (thin in x)         2       0.22     0.129     0.216        2-2
  vertical (thin in x)         4       0.44     0.255     0.420        2-2
  vertical (thin in x)         9       0.98     0.573     0.831        2-3
  vertical (thin in x)        18       1.97     0.933     1.000        3-4
  vertical (thin in x)        36       3.94     1.000     1.000        5-6
horizontal (thin in y)         1       0.22     0.129     0.224        2-2
horizontal (thin in y)         2       0.45     0.259     0.431        2-2
horizontal (thin in y)         4       0.90     0.506     0.780        2-3
horizontal (thin in y)         9       2.02     0.945     1.000        3-4
horizontal (thin in y)        18       4.03     1

- **Kết quả**: phép resize là bicubic với `antialias=True`; mỗi điểm ảnh ra ứng với 9.14 điểm ảnh vào theo chiều ngang và 4.46 theo chiều dọc. Đỉnh còn lại của vạch trắng (giá trị trắng = 1), tính theo khoảng min–max trên 10 vị trí:
  - vạch dọc (mảnh theo chiều ngang): rộng 1 px còn 0.067–0.110; 4 px còn 0.255–0.420; 9 px còn 0.573–0.831; từ 18 px trở lên còn gần đủ (0.933–1.000).
  - vạch ngang (mảnh theo chiều dọc): rộng 1 px còn 0.129–0.224; 4 px còn 0.506–0.780; từ 9 px trở lên còn 0.945–1.000.
- **Ý nghĩa**: với vạch hẹp hơn một điểm ảnh ra, đỉnh còn lại xấp xỉ bằng độ rộng chia hệ số co. Ví dụ vạch dọc 1 px: 1/9.14 ≈ 0.11, khớp với peak_max. Peak_min thấp hơn vì vạch rơi vào giữa hai điểm ảnh ra và bị chia đôi. Vạch dọc mất tương phản nhanh gấp khoảng 2 lần vạch ngang (9.14 so với 4.46), do ảnh bị nén ngang mạnh hơn (`00-data-gc10` §2.1). [SUY LUẬN] Chi tiết của vết lỗi mảnh hơn khoảng 18 px theo chiều ngang (hoặc 9 px theo chiều dọc) ở ảnh gốc bị mờ ngay từ bước 01, trước khi DINO nhìn thấy. Phép đo này dùng ảnh tổng hợp đen trắng; vết lỗi thật có tương phản thấp hơn và hình dạng khác, nên mức mất theo từng lớp cần đo trên ảnh thật (dự kiến ở `01a`).

### 2.2 NEU: phóng 200 → 256 rồi cắt giữa 224

**Cách làm**: in vùng ảnh gốc được giữ lại sau khi cắt (đổi toạ độ ngược lại: chia cho hệ số phóng 256/200 = 1.28), và cỡ một patch 16 px của ViT tính bằng điểm ảnh gốc cho cả hai bộ. Sau đó lặp lại phép đo vạch của 2.1 trên nền xám (128), để thấy cả phần vượt đỉnh lẫn phần tối hai bên vạch (ringing). Ringing sinh ra do nhân bicubic có hai thuỳ âm. Phóng to thì không có lọc chống răng cưa làm mờ như thu nhỏ.

In [5]:
# cell 4: NEU (200x200): Resize 256 then CenterCrop 224 (dino-neudet.py default): the part of the image kept, and a thin line
N, R, C = 200, 256, 224
tf = torchvision.transforms.Compose([torchvision.transforms.Resize(R, 3), torchvision.transforms.CenterCrop(C)])
print(tf)
z, off = R / N, (R - C) / 2
print(f"zoom x{z:.2f} | kept original pixels [{off / z:.1f}, {(off + C) / z:.1f}) of [0, {N}) on each axis "
      f"= {C / R:.1%} per axis, {(C / R) ** 2:.1%} of the area")
print(f"one 16-px ViT patch covers {16 / z:.1f} x {16 / z:.1f} original px (NEU), {16 * W / S:.0f} x {16 * H / S:.0f} (GC10)")
rows = []
for width in (1, 2, 4):
    peak, under = [], []
    for p in range(N // 2, N // 2 + 5):                         # 5 positions (phases)
        a = np.full((N, N, 3), 128, np.uint8)                   # gray background: overshoot and undershoot both visible
        a[:, p:p + width] = 255
        prof = np.asarray(tf(Image.fromarray(a)))[C // 2, :, 0].astype(float)
        peak.append((prof.max() - 128) / 127)                   # 1 = full line contrast kept
        under.append((128 - prof.min()) / 127)                  # dark ringing next to the line (negative bicubic lobes)
    rows.append(dict(width_px=width, width_out=round(width * z, 2), peak_min=round(min(peak), 3),
                     peak_max=round(max(peak), 3), undershoot_max=round(max(under), 3)))
print(pd.DataFrame(rows).to_string(index=False))

Compose(
    Resize(size=256, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
)
zoom x1.28 | kept original pixels [12.5, 187.5) of [0, 200) on each axis = 87.5% per axis, 76.6% of the area
one 16-px ViT patch covers 12.5 x 12.5 original px (NEU), 146 x 71 (GC10)
 width_px  width_out  peak_min  peak_max  undershoot_max
        1       1.28     0.787       1.0           0.071
        2       2.56     1.000       1.0           0.071
        4       5.12     1.000       1.0           0.071


- **Kết quả**: NEU phóng ×1.28 (200 → 256) rồi cắt giữa 224, giữ điểm ảnh gốc [12.5, 187.5) trên mỗi trục: 87.5% mỗi chiều, 76.6% diện tích. Một patch 16 px của ViT phủ 12.5 × 12.5 điểm ảnh gốc ở NEU, nhưng 146 × 71 ở GC10. Vạch trên nền xám:
  - rộng 1 px: đỉnh còn 0.787–1.0;
  - rộng 2 và 4 px: còn đủ (1.0);
  - mọi độ rộng: phần tối hai bên vạch (undershoot) tới 0.071.
- **Ý nghĩa**: phóng to hầu như giữ nguyên tương phản. Ở NEU, chỉ vạch 1 px mất tới khoảng 21% khi rơi lệch pha; còn GC10 làm mất tới 89–93% ở cùng độ rộng (2.1). Mỗi patch của GC10 phủ diện tích ảnh gốc gấp khoảng 66 lần NEU (146 × 71 so với 12.5 × 12.5, tự nhân). Bicubic tạo một viền tối nhỏ cạnh mép sáng (7% tương phản), một chi tiết giả có ở mọi ảnh đã resize. NEU mất 12.5 px ở mỗi mép; hộp lỗi bị cắt một phần, nhưng không hộp nào mất hẳn (`00-data-neu` §2.3).

## 3. Chuẩn hoá

### 3.1 ToTensor và Normalize trên một ảnh thật

`ToTensor` đổi ảnh 8 bit về [0, 1], dạng kênh × cao × rộng. `Normalize` biến mỗi kênh c thành x_c = (g − μ_c) / σ_c, với μ = (0.485, 0.456, 0.406), σ = (0.229, 0.224, 0.225), là trung bình và độ lệch chuẩn của ImageNet. Ảnh xám có ba kênh RGB bằng nhau, nên sau Normalize ba kênh khác nhau chỉ bởi một phép affine của cùng một ảnh xám g.

**Cách làm**: lấy ảnh S0 đầu tiên của mỗi bộ (theo `data_io.images`), cho qua đúng phần resize của 2.1/2.2 rồi ToTensor + Normalize. In: ảnh gốc có ba kênh bằng nhau không; shape, min, max, trung bình từng kênh của đầu vào DINO; và sau khi đảo ngược Normalize (g = x · σ + μ), ba kênh có trùng nhau không.

In [6]:
# cell 5: ToTensor + Normalize on the first S0 image of each dataset: a gray image becomes 3 channels that differ by an affine map
import torch
import data_io
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
norm = torchvision.transforms.Compose([torchvision.transforms.ToTensor(), torchvision.transforms.Normalize(MEAN, STD)])
RESIZE = {"gc10": resize, "neudet": tf}                         # the resize of each dataset (cells 3 and 4)
m, s = torch.tensor(MEAN)[:, None, None], torch.tensor(STD)[:, None, None]
for ds, rs in RESIZE.items():
    imgs = data_io.images(ds)
    p = imgs.loc[imgs["role"] == "S0", "path"].iloc[0]
    im = Image.open(p)
    a = np.asarray(im.convert("RGB"))
    x = norm(rs(im.convert("RGB")))                             # 3 x 224 x 224: what DINO receives
    g = x * s + m                                               # Normalize undone
    print(f"{ds}: {Path(p).name} | file mode {im.mode}, {im.size[0]}x{im.size[1]} | 3 RGB channels equal: "
          f"{bool((a[..., 0] == a[..., 1]).all() and (a[..., 1] == a[..., 2]).all())}")
    print(f"  DINO input {tuple(x.shape)} {x.dtype} | per channel min {[round(v, 2) for v in x.amin((1, 2)).tolist()]} "
          f"max {[round(v, 2) for v in x.amax((1, 2)).tolist()]} mean {[round(v, 2) for v in x.mean((1, 2)).tolist()]}")
    print(f"  undo Normalize: max |channel difference| {float((g - g[0]).abs().max()):.1e} | gray mean {float(g[0].mean()):.3f}")

gc10: img_06_425613700_00781.jpg | file mode L, 2048x1000 | 3 RGB channels equal: True
  DINO input (3, 224, 224) torch.float32 | per channel min [-1.98, -1.9, -1.67] max [-0.53, -0.41, -0.18] mean [-1.0, -0.89, -0.67]
  undo Normalize: max |channel difference| 3.0e-08 | gray mean 0.256
neudet: crazing_101.jpg | file mode RGB, 200x200 | 3 RGB channels equal: True
  DINO input (3, 224, 224) torch.float32 | per channel min [-1.36, -1.27, -1.04] max [2.25, 2.43, 2.64] mean [0.43, 0.57, 0.79]
  undo Normalize: max |channel difference| 6.0e-08 | gray mean 0.584


- **Kết quả**:
  - GC10 (`img_06_425613700_00781.jpg`): file mode L (một kênh), 2048×1000. Sau khi đổi sang RGB, ba kênh bằng nhau. Đầu vào DINO (3, 224, 224) float32; trung bình từng kênh [-1.0, -0.89, -0.67]; khoảng giá trị [-1.98, -0.53] ở kênh đầu. Đảo ngược Normalize thì ba kênh lệch nhau tối đa 3.0e-08; trung bình xám 0.256.
  - NEU (`crazing_101.jpg`): file mode RGB với ba kênh bằng nhau, 200×200. Trung bình từng kênh [0.43, 0.57, 0.79]; khoảng [-1.36, 2.25] ở kênh đầu. Đảo ngược thì lệch 6.0e-08; trung bình xám 0.584.
- **Ý nghĩa**: DINO nhận ba kênh chỉ là ba bản dịch và co giãn của một ảnh xám (lệch ở mức sai số float32), nên không có thông tin màu nào. Ảnh GC10 này tối (xám trung bình 0.256), nên sau Normalize mọi điểm ảnh đều âm, trung bình khoảng −0.9, xa mức 0 của ImageNet. Ảnh NEU này sáng hơn (0.584), trung bình dương. Đây mới là một ảnh mỗi bộ; 3.2 đo trên toàn bộ S0.

### 3.2 Thống kê đầu vào trên toàn bộ ảnh S0, so với ImageNet

Normalize dùng trung bình và độ lệch chuẩn của ImageNet, nên với ảnh ImageNet đầu vào có trung bình ≈ 0, độ lệch chuẩn ≈ 1 ở mỗi kênh. Câu hỏi: với ảnh thép của từng bộ thì sao?

**Cách làm**: đọc mọi ảnh S0 (GC10 764, NEU 576), qua đúng phần resize của bộ đó, lấy ảnh xám g ∈ [0, 1]. Tính:
- trung bình μ_g và độ lệch chuẩn σ_g trên mọi điểm ảnh;
- khoảng trung bình từng ảnh (phân vị 5–95);
- độ lệch chuẩn trong ảnh (trung vị), đại diện cho độ tương phản của mỗi ảnh.

Từ đó suy ra đầu vào sau Normalize ở kênh c: trung bình (μ_g − μ_c) / σ_c, độ lệch chuẩn σ_g / σ_c.

In [7]:
# cell 6: input statistics over all S0 images of each dataset, against the ImageNet statistics used by Normalize
import time
rows = []
for ds, rs in RESIZE.items():
    t = time.time()
    imgs = data_io.images(ds)
    paths = imgs.loc[imgs["role"] == "S0", "path"]
    g = np.stack([np.asarray(rs(Image.open(p).convert("RGB")))[..., 0] / 255 for p in paths])   # n x 224 x 224, gray in [0, 1]
    mu, sd = g.mean(), g.std()
    rows.append(dict(dataset=ds, images=len(g), seconds=round(time.time() - t, 1), gray_mean=round(mu, 3), gray_std=round(sd, 3),
                     image_mean_p5_p95=f"{np.percentile(g.mean((1, 2)), 5):.2f}-{np.percentile(g.mean((1, 2)), 95):.2f}",
                     image_std_median=round(float(np.median(g.std((1, 2)))), 3),
                     input_mean=[round((mu - m) / s, 2) for m, s in zip(MEAN, STD)],
                     input_std=[round(sd / s, 2) for s in STD]))
print(pd.DataFrame(rows).to_string(index=False))
print("ImageNet after Normalize: mean [0, 0, 0], std [1, 1, 1] (by construction)")

dataset  images  seconds  gray_mean  gray_std image_mean_p5_p95  image_std_median           input_mean          input_std
   gc10     764     13.0      0.317     0.148         0.14-0.46             0.065 [-0.74, -0.62, -0.4] [0.65, 0.66, 0.66]
 neudet     576      1.1      0.480     0.204         0.26-0.71             0.101  [-0.02, 0.11, 0.33] [0.89, 0.91, 0.91]
ImageNet after Normalize: mean [0, 0, 0], std [1, 1, 1] (by construction)


- **Kết quả**:
  - GC10 (764 ảnh, 13.0 s): xám trung bình 0.317, độ lệch chuẩn 0.148. Trung bình từng ảnh trải từ 0.14 đến 0.46 (phân vị 5–95). Độ lệch chuẩn trong ảnh (trung vị) chỉ 0.065. Sau Normalize: trung bình [-0.74, -0.62, -0.4], độ lệch chuẩn [0.65, 0.66, 0.66].
  - NEU (576 ảnh, 1.1 s): xám trung bình 0.480, độ lệch chuẩn 0.204. Trung bình từng ảnh 0.26–0.71; độ lệch chuẩn trong ảnh 0.101. Sau Normalize: trung bình [-0.02, 0.11, 0.33], độ lệch chuẩn [0.89, 0.91, 0.91].
- **Ý nghĩa**:
  - Đầu vào NEU gần với chuẩn ImageNet (trung bình ≈ 0, độ lệch chuẩn ≈ 0.9).
  - Đầu vào GC10 lệch xa: tối hơn (trung bình −0.4 đến −0.74) và ít biến thiên hơn (0.65).
  - Ở GC10, phần lớn độ biến thiên nằm giữa các ảnh chứ không nằm trong từng ảnh. Tự tính: 0.065² / 0.148² ≈ 0.19, tức độ biến thiên trong một ảnh điển hình chỉ khoảng 19% tổng phương sai.
  - [SUY LUẬN] Hai hệ quả:
    - GC10 xa miền ImageNet hơn NEU, một lý do fine-tune có thể có ích hơn ở GC10.
    - Độ sáng chung của cả ảnh (thay đổi theo camera, `00-data-gc10` §2.2) lấn át tương phản của vết lỗi bên trong ảnh, nên đặc trưng toàn ảnh dễ gom ảnh theo độ sáng.

## 4. Kiểm đầu cuối

### 4.1 Trích lại đặc trưng `fz` từ ảnh gốc

Nếu mục 1–3 mô tả đúng tiền xử lý, thì dựng lại transform (resize, ToTensor, Normalize) và cho qua đúng DINO gốc phải ra lại các hàng đã lưu trong file đặc trưng `fz`.

**Cách làm**: lấy 128 ảnh train đầu tiên theo thứ tự hàng của file đặc trưng. Đây đúng là batch đầu khi trích (DataLoader `shuffle=False`, batch 128). Cho qua `facebook/dino-vitb16` (Hugging Face, cache cục bộ) ở chế độ eval, không gradient, lấy token CLS của `last_hidden_state` như `--no_finetune`. So với hàng 0–127 của `features-dino-vitb16.npy` của run `fz`, cho cả GC10 và NEU.

**Quy tắc đạt** (chốt trước khi chạy): với mỗi bộ, cosine nhỏ nhất giữa hàng trích lại và hàng đã lưu ≥ 0.9999. Trùng từng bit là tốt hơn, nhưng không bắt buộc: phép chập (convolution) trên GPU có thể chọn thuật toán khác giữa hai lần chạy, nên kết quả có thể lệch ở chữ số cuối. Không đạt thì dừng notebook và chẩn đoán trước khi sang 02.

In [8]:
# cell 7: end-to-end check: re-extract the first batch (128 train images) with frozen DINO, as dino-neudet.py --no_finetune
#         did (DataLoader shuffle=False, batch 128), and compare with rows 0-127 of the stored fz features
os.environ.setdefault("HF_HUB_OFFLINE", "1")                    # model from the local cache, as scripts/run_phase1_neu.sh
from transformers import ViTModel
FZ = {"gc10": "20261005_013042__gc10_fz_cls5first_img80pct", "neudet": "20261005_014929__neu_fz_cls3first_img80pct"}
dev = "cuda" if torch.cuda.is_available() else "cpu"
model = ViTModel.from_pretrained("facebook/dino-vitb16").to(dev).eval()
print("device", dev, torch.cuda.get_device_name(0) if dev == "cuda" else "", "| torch", torch.__version__)
for ds, run in FZ.items():
    imgs = data_io.images(ds)
    paths = imgs[imgs["part"] == "train"].sort_values("row")["path"].iloc[:128]
    tfm = torchvision.transforms.Compose([RESIZE[ds], norm])   # the transform of 1.1: resize, ToTensor, Normalize
    x = torch.stack([tfm(Image.open(p).convert("RGB")) for p in paths]).to(dev)
    with torch.no_grad():
        f = model(x).last_hidden_state[:, 0].cpu().numpy()     # CLS token, as extract(..., cls=True)
    ref = np.load(f"features/{run}/dino-vitb16/features-dino-vitb16.npy")[:128]
    cos = (f * ref).sum(1) / np.linalg.norm(f, axis=1) / np.linalg.norm(ref, axis=1)
    print(f"{ds}: {run} | rows 0-127 | bit-identical {np.array_equal(f, ref)} | max|diff| {np.abs(f - ref).max():.2e} "
          f"(max|value| {np.abs(ref).max():.2f}) | min cosine {cos.min():.7f}")

/home/rito/miniforge3/envs/vbcgcd/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of ViTModel were not initialized from the model checkpoint at facebook/dino-vitb16 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


device cuda NVIDIA GeForce RTX 4060 Laptop GPU | torch 2.4.0+cu121
gc10: 20261005_013042__gc10_fz_cls5first_img80pct | rows 0-127 | bit-identical True | max|diff| 0.00e+00 (max|value| 16.23) | min cosine 0.9999999
neudet: 20261005_014929__neu_fz_cls3first_img80pct | rows 0-127 | bit-identical True | max|diff| 0.00e+00 (max|value| 18.75) | min cosine 0.9999999


- **Kết quả**: trên GPU RTX 4060 (torch 2.4.0+cu121), 128 hàng trích lại trùng từng bit (bit-identical) với hàng 0–127 của file `fz` ở cả hai bộ: max|diff| = 0, cosine nhỏ nhất 0.9999999 (sai số làm tròn của phép tính cosine). Trị tuyệt đối lớn nhất: 16.23 (GC10), 18.75 (NEU). Cảnh báo "pooler newly initialized" là bình thường: `--no_finetune` lấy CLS, không dùng pooler (D8).
- **Ý nghĩa**: đạt quy tắc đã chốt, và còn đạt mức mạnh nhất (trùng từng bit). Mục 1–3 mô tả đúng và đủ tiền xử lý đã tạo ra đặc trưng `fz`: không có bước ẩn nào, và môi trường hiện tại (torch, transformers, Pillow, driver) vẫn tái tạo được đặc trưng của ngày 5/10. Đặc trưng `ft` cần checkpoint fine-tune nên kiểm ở 03.

## Summary (EN)

1. **One transform for fine-tuning and extraction.** All 8 phase-1 runs (GC10 and NEU, `fz` and `ft` r0–r2) ran at commit `97c57ee`, and `feature_extractor/` is unchanged since. GC10 uses `--resize full` (bicubic to 224×224); NEU uses Resize 256 + CenterCrop 224. Both then apply ToTensor and the ImageNet Normalize. Fine-tuning reads the same `tfm` and uses no augmentation. The `transform` defined in `finetune_dino` is never read (§1.1–1.2).
2. **GC10 loses thin detail at step 01.** The resize squeezes GC10 by 9.14 horizontally and 4.46 vertically with antialiased bicubic. A 1 px bright line keeps 0.067–0.110 of its contrast if vertical and 0.129–0.224 if horizontal. Full contrast needs about 18 px and 9 px. NEU is upsampled ×1.28 and keeps ≥ 0.787 of a 1 px line. One ViT patch covers 146×71 original pixels on GC10 and 12.5×12.5 on NEU (§2.1–2.2). [inference] Defect detail thinner than about 18 px horizontally is blurred before DINO sees it; per-class loss on real defects is not measured here.
3. **GC10 inputs are far from ImageNet statistics.** Gray images become three affine copies of one channel, so DINO gets no color. After Normalize, GC10 S0 inputs have mean −0.74 to −0.40 and std 0.65–0.66 per channel, against 0 and 1 for ImageNet; NEU has −0.02 to 0.33 and 0.89–0.91. In GC10 the median within-image std is 0.065 against 0.148 over all pixels (§3.1–3.2). [inference] Brightness differences between images dominate the defect contrast inside an image.
4. **The description is exact.** Re-extracting the first batch (128 train images) with frozen DINO reproduces rows 0–127 of the stored `fz` features bit for bit, for GC10 and NEU (§4.1).